#### **Autocorrelation and statistical significance in MD simulations** <br> Joshua Pajak, Ph.D. joshua.pajak@umassmed.edu

This Jupyter notebook is intended to show the reader the dangers of saving trajectories too frequently in molecular dynamics simulations. We will play with two toy datasets, *distance1* and *distance2*. 

Imagine this scnario: you are trying to understand if a drug molecule that your lab designed moves two catalytically important residues farther apart. So you have run extensive MD simulations, and measured the distances between these residues in *VMD*. Now you are plotting these values, and would like to know whether they are statistically distinct distributions. What could go wrong? 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# --- load in the data into a PD dataframe ---
d1 = pd.read_csv('distance.tsv', sep='\t')
d2 = pd.read_csv('distance2.tsv', sep='\t')

df = pd.DataFrame({
    'distance1': d1['distance'],
    'distance2': d2['distance']
})

In [ ]:
# --- do the eye test ---
sns.set_style('whitegrid')
sns.kdeplot(df, fill=True, palette='hls')
plt.title("Distributions of Distances")
plt.xlabel("Distance (Å)")

In [ ]:
# --- Now run a t-test and report the p-value. Done! ---
t_stat, p_full = stats.ttest_ind(df['distance1'], df['distance2'])
print(f"Full Dataset (N={len(df)}):")
print(f"p-value: {p_full:.2e}")

Wow, that p-value is **remarkably** small. You ran good simulations! Pat yourself on the back.

Your labmate wants to play with your data, but your trajectory file is too big to send to her. So instead, you save every 100th frame and send the pruned dataset. She repeats your analysis:

In [ ]:
stride = 100 
df_stride = df[::stride]
t_stat_sub, p_sub = stats.ttest_ind(df_stride['distance1'], df_stride['distance2'])
print(f"\nSubsampled Dataset (N={len(df_stride)}, every {stride}th frame):")
print(f"p-value: {p_sub:.4f}")

This p-value is much different! Why? Because MD data is a **time series**, you need to be careful about autocorrelation. Autocorrelation (self-correlation) is the idea that for small changes in time, your system can't move very far away from the previous timepoint. Thus, your data is full of "psuedo-replication" which can artifically inflate p-values. 

We are often taught to report the standard error of the mean, which is given by: $$SEM = \frac{\sigma}{\sqrt{N}}$$

Instead, we need to consider the **effective** sample size: $$SEM = \frac{\sigma}{\sqrt{N_{eff}}}$$

where $N_{eff}$ is given by: $$N_{eff}=\frac{N_{total}}{1+2\tau}$$
and $\tau$ is the **autocorrelation time.** This is typically the timestep required for the autocorrelation value to cross ~37% or $\frac{1}{e}$.

Ok, so what is autocorrelation? It is the Pearson correlation between values of a function at different lag times. In math: $$\rho_k = \frac{\sum_{t=1}^{N-k} (X_t - \mu)(X_{t+k} - \mu)}{\sum_{t=1}^{N} (X_t - \mu)^2}$$
where $\mu$ is the average value of your series.


In [ ]:
# --- Pandas has a built-in autocorrelation method for all DFs --- 
def get_acf(series, max_lag=250):
    return [series.autocorr(lag=i) for i in range(max_lag)]

plt.plot(get_acf(df['distance1']), label='ACF Distance1')
plt.plot(get_acf(df['distance2']), label='ACF Distance2')
plt.axhline(0, color='black', linestyle='--')
plt.title("Autocorrelation Function (ACF)")
plt.xlabel("Lag (Frames)")
plt.ylabel("Correlation")
plt.legend()
plt.show()

**Understanding Statistical Inefficiency ($g$)**.
Before, we talked about the "effective sample size." The divisor is often called your "statistical inefficiency" and denoted as $g$. $$N_{eff} = \frac{N}{g}$$ 

In reality, the correlation time is given by:
$$\tau = \int_{0}^{\infty} \rho(t) dt$$
However, because we have finite data we will approximate this integral with a trapezoidal summation:
$$g \approx 1 + 2 \sum_{k=1}^{T} \rho_k$$
In practice, we have to normalize by the fact that for longer lag times we have fewer steps that we can calcualte:
$$g = 1 + 2 \sum_{k=1}^{N-1} \left( 1 - \frac{k}{N} \right) \rho_k$$

The $\left( 1 - \frac{k}{N} \right)$ term looks like a complicated correction. But, when you calculate the correlation at a lag of $k$, you are comparing pairs of frames.If your trajectory has $N$ frames, and you shift it by a lag of $k$, you only have $N-k$ pairs left to compare. For example: If $N=100$ and your lag $k=90$, you only have 10 data points ($100-90$) to calculate that correlation. This makes the estimate at high lags very noisy and unreliable.The term $\left( 1 - \frac{k}{N} \right)$ is a weight that reflects how much of the trajectory is actually being "sampled" at that specific lag. As the lag $k$ approaches the total time $N$, the weight goes to zero because we have run out of data to correlate.

In [ ]:
def calculate_statistical_inefficiency(data, max_lag=500):
    """
    Calculates the statistical inefficiency (g).
    Returns the 'stride' needed for independent samples.
    """
    # --- Get ACF values ---
    acf = [data.autocorr(lag=i) for i in range(max_lag)]
    
    # --- Sum the ACF values until they hit the first zero-crossing ---
    g = 1.0
    for i in range(1, len(acf)):
        if acf[i] <= 0: # Stop once we hit the first zero-crossing
            break
        g += 2 * acf[i]
        
    return g

# --- Application ---
g_val = calculate_statistical_inefficiency(df['distance1'])
stride_ideal = int(np.ceil(g_val))

print(f"Calculated Statistical Inefficiency (g): {g_val:.2f}")
print(f"Recommended Stride for Independence: {stride_ideal}")

# --- Final Statistical Test ---
df_ideal = df[::stride_ideal]
t_final, p_final = stats.ttest_ind(df_ideal['distance1'], df_ideal['distance2'])

print(f"\n--- Final Results (using stride {stride_ideal}) ---")
print(f"Independent Samples (N): {len(df_ideal)}")
print(f"Corrected p-value: {p_final:.4f}")

### **Homework for the reader**
1. Write your own autocorrelation function rather than using Pandas' built-in function.
2. When the statistical inefficiency of one dataset is 51, and the other dataset is 99, what should you do?